# Домашняя работа 1 — доли исходов HMDA

## Подготовка

In [1]:
from pathlib import Path
import sys
import pandas as pd

print("Python:", sys.executable)
print("pandas:", pd.__version__)
print("Рабочая папка:", Path.cwd())
DATA = Path("../data")
assert DATA.is_dir(), "Рядом с папкой ноутбука должна находиться data/"

Python: /usr/bin/python3
pandas: 3.0.2
Рабочая папка: /home/claude/repo/data-analysis-course-main/lessons/01-introduction/work


## Основной набор decisions.csv

In [2]:
home_decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
home_outcomes = home_decisions["action_taken"]
home_n = len(home_decisions)
assert home_n > 0
assert home_outcomes.isna().sum() == 0
assert home_outcomes.isin([1, 2, 3]).sum() == home_n
print(home_decisions.shape)
home_outcomes.value_counts(dropna=False)

(6830, 7)


action_taken
1    5956
3     691
2     183
Name: count, dtype: int64

## H1. Доля отказов

In [3]:
home_denied_flag = home_outcomes.eq(3)
home_denied = home_denied_flag.sum()
home_denial_rate = home_denied / home_n

print("Условие: action_taken == 3 (отказ)")
print("Числитель (отказы):", home_denied)
print("Знаменатель (все записи с исходами 1-3):", home_n)
print("Доля отказов:", home_denial_rate)
print(f"Доля отказов: {100 * home_denial_rate:.2f}%")

Условие: action_taken == 3 (отказ)
Числитель (отказы): 691
Знаменатель (все записи с исходами 1-3): 6830
Доля отказов: 0.10117130307467057
Доля отказов: 10.12%


In [4]:
home_approved = home_outcomes.isin([1, 2]).sum()

assert home_denied + home_approved == home_n
assert 0 <= home_denial_rate <= 1
print("Одобрено (коды 1 и 2):", home_approved)
print("Отказано + одобрено =", home_denied + home_approved, "= home_n =", home_n)

Одобрено (коды 1 и 2): 6139
Отказано + одобрено = 6830 = home_n = 6830


## H2. Одобрение без принятия предложения

In [6]:
home_not_accepted = home_outcomes.eq(2).sum()
home_not_accepted_rate = home_not_accepted / home_n

print("Условие: action_taken == 2 (одобрено, предложение не принято)")
print("Числитель:", home_not_accepted)
print("Знаменатель:", home_n)
print(f"Доля одобрений без принятия предложения: {100 * home_not_accepted_rate:.2f}%")

Условие: action_taken == 2 (одобрено, предложение не принято)
Числитель: 183
Знаменатель: 6830
Доля одобрений без принятия предложения: 2.68%


In [7]:
home_issued = home_outcomes.eq(1).sum()

# Связь 1: по исходным количествам (без округления процентов)
assert home_approved - home_issued == home_not_accepted
print("Одобрено - выдано =", home_approved - home_issued, "= код 2:", home_not_accepted)

Одобрено - выдано = 183 = код 2: 183


In [8]:
home_approval_rate = home_approved / home_n
home_origination_rate = home_issued / home_n
gap_pp = 100 * (home_approval_rate - home_origination_rate)

print(f"Доля одобрений:  {100 * home_approval_rate:.2f}%  ({home_approved} / {home_n})")
print(f"Доля выдач:      {100 * home_origination_rate:.2f}%  ({home_issued} / {home_n})")
print(f"Разность: {gap_pp:.2f} процентного пункта")

# Связь 2: разность долей равна доле исхода 2
assert abs(gap_pp - 100 * home_not_accepted_rate) < 1e-9
print(f"Доля кода 2: {100 * home_not_accepted_rate:.2f}% - совпадает с разностью")

Доля одобрений:  89.88%  (6139 / 6830)
Доля выдач:      87.20%  (5956 / 6830)
Разность: 2.68 процентного пункта
Доля кода 2: 2.68% - совпадает с разностью


## H3. Объяснение инструментов

## Дополнительная часть

## Исходный снимок для H+1

In [10]:
raw = pd.read_csv(
    DATA / "hmda_2023_dc_raw.csv",
    dtype={"lei": "string", "county_code": "string"},
)
raw_outcomes = raw["action_taken"]
raw_n = len(raw)
assert raw_n > 0
print(raw.shape)
raw_outcomes.value_counts(dropna=False)

(8575, 99)


action_taken
1    5956
4    1585
3     691
2     183
5     160
Name: count, dtype: int64

## H+1. Доли выдач в двух наборах

In [11]:
raw_issued = raw_outcomes.eq(1).sum()
raw_origination_rate = raw_issued / raw_n

home_origination_rate = home_issued / home_n

print(f"Выдачи среди всех строк raw:       {raw_issued} / {raw_n} = {100 * raw_origination_rate:.2f}%")
print(f"Выдачи среди home_decisions (1-3): {home_issued} / {home_n} = {100 * home_origination_rate:.2f}%")

assert raw_issued == home_issued
print("Количество выдач в двух файлах совпадает:", raw_issued)

Выдачи среди всех строк raw:       5956 / 8575 = 69.46%
Выдачи среди home_decisions (1-3): 5956 / 6830 = 87.20%
Количество выдач в двух файлах совпадает: 5956


## Текстовые коды для H+2

In [13]:
text_example = pd.read_csv(
    DATA / "reading_example.csv", dtype={"action_taken": "string"}
)
text_example["action_taken"]

0    1
1    1
2    2
3    2
4    3
5    3
Name: action_taken, dtype: string

## H+2. Сравнение текста с числом и текстом

In [14]:
text_eq_number = text_example["action_taken"].eq(1)
text_eq_text = text_example["action_taken"].eq("1")

print("eq(1):")
print(text_eq_number)
print("eq(\"1\"):")
print(text_eq_text)
print("Количество True: eq(1) =", text_eq_number.sum(), "; eq(\"1\") =", text_eq_text.sum())

eq(1):
0    False
1    False
2    False
3    False
4    False
5    False
Name: action_taken, dtype: boolean
eq("1"):
0     True
1     True
2    False
3    False
4    False
5    False
Name: action_taken, dtype: boolean
Количество True: eq(1) = 0 ; eq("1") = 2
